In [ ]:
import pandas as pd

preds = pd.read_csv("dSets/test_predictions.csv")

DECISION_THRESHOLD = 0.16

def decide_action(probability: float, threshold: float = DECISION_THRESHOLD) -> str:
    """
    Turns a model probability into one of two bounded actions.
    This is the entire 'decision menu' the agent is allowed to choose from -
    nothing outside these two strings can ever be returned.
    """
    if probability >= threshold:
        return "flag_for_review"
    else:
        return "auto_approve"

preds["action"] = preds["log_reg_prob"].apply(decide_action)

print("=== Action distribution across test set ===")
print(preds["action"].value_counts())
print()

# Sanity check: does this match what we found in Step 2.5?
# (512 flagged out of 1000 at threshold 0.16)
n_flagged = (preds["action"] == "flag_for_review").sum()
print(f"Orders flagged for review: {n_flagged} / {len(preds)}")

preds.to_csv("dSets/orders_with_actions.csv", index=False)
print("\nSaved: dSets/orders_with_actions.csv")

=== Action distribution across test set ===
action
flag_for_review    512
auto_approve       488
Name: count, dtype: int64

Orders flagged for review: 512 / 1000

Saved: orders_with_actions.csv


In [18]:
import pandas as pd
import numpy as np

X_trainpool = pd.read_csv("dSets/X_trainpool.csv")
y_trainpool = pd.read_csv("dSets/y_trainpool.csv").squeeze()
X_test = pd.read_csv("dSets/X_test_FINAL.csv")
preds = pd.read_csv("dSets/orders_with_actions.csv")

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

log_reg_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
log_reg_final.fit(X_trainpool, y_trainpool)

scaler = log_reg_final.named_steps["scaler"]
model = log_reg_final.named_steps["model"]
feature_names = X_trainpool.columns

def readable_label(feature_name: str, scaled_value: float = None, raw_value=None) -> str:
    manual_overrides = {
        "discount_pct": "large discount",
        "size_variant_flag": "size-dependent item",
        "is_first_time_buyer": "first-time buyer",
        "customer_past_return_rate": "customer's return history",
        "customer_past_orders": "customer's order count",
        "days_to_deliver": "delivery time",
    }

    if feature_name == "order_value":
        return "unusually low order value" if scaled_value is not None and scaled_value < 0 else "unusually high order value"

    if feature_name == "has_return_history":
        return "no prior return history available" if raw_value == 0 else "established return history"

    if feature_name in manual_overrides:
        return manual_overrides[feature_name]
    if feature_name.startswith("category_"):
        return f"{feature_name.replace('category_', '')} category"
    if feature_name.startswith("payment_method_"):
        return f"{feature_name.replace('payment_method_', '')} payment"
    if feature_name.startswith("delivery_pincode_tier_"):
        return f"{feature_name.replace('delivery_pincode_tier_', '')} delivery area"
    if feature_name.startswith("time_of_day_ordered_"):
        return f"ordered in the {feature_name.replace('time_of_day_ordered_', '')}"

    return feature_name

def explain_order(feature_row: pd.DataFrame, top_n: int = 3) -> str:
    scaled_values = scaler.transform(feature_row)[0]
    contributions = scaled_values * model.coef_[0]
    raw_values = feature_row.iloc[0]

    contrib_series = pd.Series(contributions, index=feature_names)
    scaled_series = pd.Series(scaled_values, index=feature_names)

    one_hot_prefixes = ("category_", "payment_method_", "delivery_pincode_tier_", "time_of_day_ordered_")

    def is_valid_driver(feat):
        if feat.startswith(one_hot_prefixes):
            return raw_values[feat] == 1
        return True

    candidates = contrib_series[contrib_series > 0.05].sort_values(ascending=False)
    candidates = candidates[[is_valid_driver(f) for f in candidates.index]]
    top_positive = candidates.head(top_n)

    reasons = [
        readable_label(feat, scaled_series[feat], raw_values[feat])
        for feat in top_positive.index
    ]

    if not reasons:
        return "No strong individual risk drivers identified."
    return "Elevated risk driven by: " + ", ".join(reasons)

explanations = []
for i in range(len(X_test)):
    row_df = X_test.iloc[[i]]
    explanations.append(explain_order(row_df))
preds["explanation"] = explanations
preds.to_csv("dSets/orders_with_explanations.csv", index=False)



In [19]:
approved_sample = preds[preds["action"] == "auto_approve"].head(3)
print("=== Sample explanations for auto-approved orders ===\n")
for idx, row in approved_sample.iterrows():
    print(f"Order {idx}: prob={row['log_reg_prob']:.3f} -> {row['action']}")
    print(f"  {row['explanation']}\n")

=== Sample explanations for auto-approved orders ===

Order 2: prob=0.029 -> auto_approve
  Elevated risk driven by: home category

Order 6: prob=0.148 -> auto_approve
  Elevated risk driven by: COD payment, large discount, customer's return history

Order 8: prob=0.073 -> auto_approve
  Elevated risk driven by: unusually low order value



In [21]:
import pandas as pd
from datetime import datetime, timezone

original_df = pd.read_csv("dSets/synthetic_orders_model.csv")

from sklearn.model_selection import train_test_split

df_prepped = pd.read_csv("dSets/synthetic_orders_prepped.csv")
X_full = df_prepped.drop(columns=["returned"])
y_full = df_prepped["returned"]

_, test_idx_df, _, _ = train_test_split(
    original_df, y_full, test_size=0.2, random_state=42, stratify=y_full
)

order_ids = test_idx_df["order_id"].values
customer_ids = test_idx_df["customer_id"].values

assert len(order_ids) == len(preds), "Mismatch: order_id count doesn't match predictions count"


audit_log = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": customer_ids,
    "risk_probability": preds["log_reg_prob"].round(4),
    "action_taken": preds["action"],
    "explanation": preds["explanation"],
    "decision_threshold_used": 0.16,
    "model_version": "logreg_v1_step2",
    "timestamp": datetime.now(timezone.utc).isoformat(),
})

print("=== Audit log sample ===")
print(audit_log.head(10).to_string(index=False))

print(f"\nTotal decisions logged: {len(audit_log)}")
print(f"Action breakdown:\n{audit_log['action_taken'].value_counts()}")

audit_log.to_csv("dSets/audit_log.csv", index=False)
print("\nSaved: dSets/audit_log.csv")

=== Audit log sample ===
                            order_id                          customer_id  risk_probability    action_taken                                                                               explanation  decision_threshold_used   model_version                        timestamp
a78fa342-513d-416e-aa3f-183912adf800 676c4a16-07aa-45fa-8ad1-17b29a2b6de4            0.4063 flag_for_review         Elevated risk driven by: size-dependent item, large discount, tier3 delivery area                     0.16 logreg_v1_step2 2026-08-23T06:30:21.573233+00:00
386793fc-d277-4d26-ade3-618038aa42c0 7f892973-babf-4935-bab7-187b59d8b023            0.2694 flag_for_review            Elevated risk driven by: COD payment, home category, unusually low order value                     0.16 logreg_v1_step2 2026-08-23T06:30:21.573233+00:00
b03d672c-0dcf-4385-a8eb-1b46a4bccc0d 30ed7aa6-3495-4b99-970b-27c038e98f99            0.0295    auto_approve                                                    

In [ ]:
import pandas as pd
import numpy as np

original_orders = pd.read_csv("dSets/synthetic_orders_model.csv")
mean_return_rate = original_orders["customer_past_return_rate"].mean()

print(f"Recomputed mean_return_rate: {mean_return_rate:.4f}")

edge_case_raw = pd.DataFrame([{
    "order_value": 1450.0,
    "discount_pct": 0.45,
    "is_first_time_buyer": 1,
    "customer_past_orders": 0,
    "customer_past_return_rate": np.nan,  
    "size_variant_flag": 1,
    "days_to_deliver": 4,
    "category": "fashion",
    "payment_method": "COD",
    "delivery_pincode_tier": "tier2",
    "time_of_day_ordered": "night",
}])

print("=== Raw edge-case input ===")
print(edge_case_raw.to_string(index=False))

edge_case_raw["has_return_history"] = edge_case_raw["customer_past_return_rate"].notna().astype(int)
edge_case_raw["customer_past_return_rate"] = edge_case_raw["customer_past_return_rate"].fillna(mean_return_rate)

categorical_cols = ["category", "payment_method", "delivery_pincode_tier", "time_of_day_ordered"]
edge_case_encoded = pd.get_dummies(edge_case_raw, columns=categorical_cols)

edge_case_encoded = edge_case_encoded.reindex(columns=X_test.columns, fill_value=0)

print("\n=== Running through the full agent pipeline ===")
try:
    prob = log_reg_final.predict_proba(edge_case_encoded)[:, 1][0]
    action = decide_action(prob)
    explanation = explain_order(edge_case_encoded)

    print(f"Risk probability: {prob:.4f}")
    print(f"Action taken:     {action}")
    print(f"Explanation:      {explanation}")
    print(f"has_return_history flag: {edge_case_encoded['has_return_history'].values[0]} "
          f"(0 = genuinely unknown, imputed with population mean)")
    print("\nResult: pipeline completed without error on missing-data input.")
except Exception as e:
    print(f"FAILURE: pipeline crashed with error: {e}")

Recomputed mean_return_rate: 0.2500
=== Raw edge-case input ===
 order_value  discount_pct  is_first_time_buyer  customer_past_orders  customer_past_return_rate  size_variant_flag  days_to_deliver category payment_method delivery_pincode_tier time_of_day_ordered
      1450.0          0.45                    1                     0                        NaN                  1                4  fashion            COD                 tier2               night

=== Running through the full agent pipeline ===
Risk probability: 0.7035
Action taken:     flag_for_review
Explanation:      Elevated risk driven by: size-dependent item, COD payment, no prior return history available
has_return_history flag: 0 (0 = genuinely unknown, imputed with population mean)

Result: pipeline completed without error on missing-data input.


In [ ]:
print("=== Re-check: sample flagged orders after has_return_history fix ===\n")
flagged_sample = preds[preds["action"] == "flag_for_review"].head(5)
for idx, row in flagged_sample.iterrows():
    print(f"Order {idx}: prob={row['log_reg_prob']:.3f} -> {row['action']}")
    print(f"  {row['explanation']}\n")

=== Re-check: sample flagged orders after has_return_history fix ===

Order 0: prob=0.406 -> flag_for_review
  Elevated risk driven by: size-dependent item, large discount, tier3 delivery area

Order 1: prob=0.269 -> flag_for_review
  Elevated risk driven by: COD payment, home category, unusually low order value

Order 3: prob=0.624 -> flag_for_review
  Elevated risk driven by: customer's return history, size-dependent item, large discount

Order 4: prob=0.202 -> flag_for_review
  Elevated risk driven by: size-dependent item, fashion category

Order 5: prob=0.178 -> flag_for_review
  Elevated risk driven by: COD payment, home category, unusually low order value

